# Smart Budget — SageMaker Endpoint

**Ticket:** DATA-1140 · **Endpoint:** `smart-budget-suggestion-endpoint`

1. **Crear endpoint** (Steps 1–2)
2. **Probar endpoint** (Steps 3–4)


In [ ]:
import boto3
import sagemaker
from sagemaker.sklearn.model import SKLearnModel
from sagemaker import Session

# get_execution_role() no está disponible en sagemaker-core 2.x
# Se deriva el rol desde la identidad actual de AWS (funciona en Studio y local)
def get_role():
    sts = boto3.client('sts')
    arn = sts.get_caller_identity()['Arn']
    if ':assumed-role/' in arn:
        account = arn.split(':')[4]
        role_name = arn.split(':assumed-role/')[1].split('/')[0]
        return f"arn:aws:iam::{account}:role/{role_name}"
    return arn

sagemaker_session = Session()
role = get_role()

print(f"Role   : {role}")
print(f"Region : {sagemaker_session.boto_region_name}")


---
## Step 1 — Crear y desplegar el endpoint

In [ ]:
import os
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())

model_artifact_uri = "s3://blossom-analytics-safe-dev-nv/smart_budget/endpoint/v1/model.tar.gz"

sk_model = SKLearnModel(
    model_data=model_artifact_uri,
    role=role,
    entry_point="inference.py",
    source_dir=str(REPO_ROOT / "src" / "api"),
    framework_version="1.2-1",
    sagemaker_session=sagemaker_session,
)

predictor = sk_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="smart-budget-suggestion-endpoint",
)

print("✅ Endpoint desplegado: smart-budget-suggestion-endpoint")


---
## Step 2 — Actualizar endpoint existente

> Solo si el endpoint ya existe y quieres actualizar el modelo.

In [ ]:
# Actualizar endpoint existente (si ya fue creado antes)
sk_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="smart-budget-suggestion-endpoint",
    update_endpoint=True,
)
print("✅ Endpoint actualizado")


---
## Step 3 — Probar el endpoint (happy path)


In [ ]:
import json

runtime = boto3.client('sagemaker-runtime', region_name=sagemaker_session.boto_region_name)
ENDPOINT_NAME = "smart-budget-suggestion-endpoint"

def invoke(payload: dict) -> dict:
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps(payload),
    )
    return json.loads(response['Body'].read().decode('utf-8'))

result = invoke({"idaccount": "EXT2", "defaultcategory": "Food & Dining", "period_id": "2026-05"})
print(json.dumps(result, indent=2))


---
## Step 4 — Validar las 3 reglas de negocio

| # | Condición | Esperado |
|---|---|---|
| Regla 1 | Cuenta no existe | Error (ModelError) |
| Regla 2 | Categoría inválida | Error (ModelError) |
| Regla 3 | Sin datos para el período | `suggested_amount: null` |


In [ ]:
import botocore

# Regla 1 — Cuenta no existe → error
try:
    invoke({"idaccount": "CUENTA_INEXISTENTE", "defaultcategory": "Groceries", "period_id": "2026-05"})
    print("❌ Regla 1 FALLÓ")
except (runtime.exceptions.ModelError, botocore.exceptions.ClientError):
    print("✅ Regla 1 OK — cuenta inexistente → error")

# Regla 2 — Categoría inválida → error
try:
    invoke({"idaccount": "EXT2", "defaultcategory": "CategoriaFalsa", "period_id": "2026-05"})
    print("❌ Regla 2 FALLÓ")
except (runtime.exceptions.ModelError, botocore.exceptions.ClientError):
    print("✅ Regla 2 OK — categoría inválida → error")

# Regla 3 — Sin datos → null
r = invoke({"idaccount": "SYN001", "defaultcategory": "Groceries", "period_id": "2026-05"})
assert r["suggested_amount"] is None
print(f"✅ Regla 3 OK — sin datos → null  ({r.get('display_label')})")


---
## ⚠️ Borrar endpoint cuando termines

Los endpoints generan costo por hora.

In [ ]:
boto3.client('sagemaker').delete_endpoint(EndpointName="smart-budget-suggestion-endpoint")
print("✅ Endpoint eliminado")
